In [ ]:
#extrahieren der canonical cluster der cdrs von unseren antikörpern
#Lizenz von PylgClassify beantragt

# Liste der interessanten PDBs in lowercase
your_pdbs = set(ab_ag_uniquesequences["pdb"].str.lower())

# Lade nur bestimmte Spalten der TSV
usecols = [
    "pdb",
    "cdrh1_cluster",
    "cdrh2_cluster",
    "cdrh3_length",
    "cdrl1_cluster",
    "cdrl2_cluster",
    "cdrl3_cluster"
]

#datei laden
pylgclassifyfile = pd.read_csv("sabdab_summary_all.tsv", sep='\t', usecols=usecols)
#das ist nicht die korrekte datei, da hier nicht die cluster der cdrs drin sind => pylgClassify database CSV

#Filtere direkt
pylgclassifyfile_filtered = pylgclassifyfile[pylgclassifyfile["pdb"].str.lower().isin(your_pdbs)].copy()

#Ergebnis
print(f"Anzahl PDBs mit Canonical Clusters in deinem Datensatz: {len(pylgclassifyfile_filtered)}")
print(pylgclassifyfile_filtered.head())

In [ ]:
#ARI, NMI oder V-measure als Quantitative Maße für die Qualität deines ESM-Clusterings im Vergleich zu Canonical Clustern
#Werte liegen zwsichen 0 (schlechte überinstimmung) und 1 (perfekte Überinstimmung)

In [ ]:
# CDR_list muss ersetzt werden mit ab_ag_annotated aufgereinigt?
# datei aus esmc
# datei mit referenz cluster finden - ascheinend auf sabdab für einzelene pdb einträge möglich aber find ich nicht?
# alternativ: PyIgClassify2 (offizielle quelle für cdr): https://dunbrack.fccc.edu/pyigclassify/

import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score
from sklearn.preprocessing import LabelEncoder

def compare_esm_vs_canonical(canonical_df, esm_df, cdr_list=["CDR-H1", "CDR-H2", "CDR-H3", "CDR-L1", "CDR-L2", "CDR-L3"]):
    """
    Vergleicht Canonical Cluster vs. ESM-Cluster für jede CDR-Region separat.
    
    Parameter:
        canonical_df: DataFrame mit Spalten ['PDB_ID', 'Chain', 'CDR', 'Canonical_Cluster']
        esm_df: DataFrame mit Spalten ['PDB_ID', 'Chain', 'CDR', 'ESM_Cluster_Label']
        cdr_list: Liste der CDRs, die verglichen werden sollen

    Rückgabe:
        DataFrame mit ARI, NMI und V-Measure pro CDR
    """
    results = []

    # Iteriere über alle gewünschten CDRs
    for cdr in cdr_list:
        # Filtere beide DataFrames auf die aktuelle CDR
        canon_cdr = canonical_df[canonical_df["CDR"] == cdr].copy()
        esm_cdr = esm_df[esm_df["CDR"] == cdr].copy()

        # Erstelle Matching-Key
        canon_cdr["key"] = canon_cdr["PDB_ID"].str.lower() + "_" + canon_cdr["Chain"]
        esm_cdr["key"] = esm_cdr["PDB_ID"].str.lower() + "_" + esm_cdr["Chain"]

        # Gemeinsame Keys
        common_keys = set(canon_cdr["key"]) & set(esm_cdr["key"])
        if len(common_keys) < 5:
            continue  # Zu wenige gemeinsame Daten → überspringen

        canon_common = canon_cdr[canon_cdr["key"].isin(common_keys)].sort_values("key")
        esm_common = esm_cdr[esm_cdr["key"].isin(common_keys)].sort_values("key")

        # Labels extrahieren
        le = LabelEncoder()
        canon_labels = le.fit_transform(canon_common["Canonical_Cluster"])
        esm_labels = esm_common["ESM_Cluster_Label"].tolist()

        # Metriken berechnen
        ari = adjusted_rand_score(canon_labels, esm_labels)
        nmi = normalized_mutual_info_score(canon_labels, esm_labels)
        vm = v_measure_score(canon_labels, esm_labels)

        results.append({
            "CDR": cdr,
            "Anzahl_PDBs": len(common_keys),
            "ARI": round(ari, 4),
            "NMI": round(nmi, 4),
            "V-Measure": round(vm, 4)
        })

    # Rückgabe als DataFrame
    return pd.DataFrame(results)


In [ ]:
#datei mit referenzclustern muss in richtiges format gebracht werden bevor die funktion darauf angewendet werden kann
#Zuordnung erfolgt nicht pauschal als "Heavy"/"Light", sondern entsprechend der Chain-Spalte aus unserem datensatz

canonical_entries = []

# Durchlaufe jede Zeile mit einer PDB-Kette in der PyIgClassify-Datei
for _, row in pyIgClassifyfile_filtered.iterrows():
    pdb_id = row["pdb"].lower()          # PDB in Kleinbuchstaben
    chain = row["chain"]                 # Kette (z. B. "H", "L", "A", "B", ...)

    # Wenn CDR-H1/2 vorhanden → zu Heavy Chain zählen
    if pd.notnull(row.get("cdrh1_cluster")):
        canonical_entries.append({
            "PDB_ID": pdb_id,
            "Chain": chain,
            "CDR": "CDR-H1",
            "Canonical_Cluster": row["cdrh1_cluster"]
        })

    if pd.notnull(row.get("cdrh2_cluster")):
        canonical_entries.append({
            "PDB_ID": pdb_id,
            "Chain": chain,
            "CDR": "CDR-H2",
            "Canonical_Cluster": row["cdrh2_cluster"]
        })

    # Wenn CDR-L1/2/3 vorhanden → zu Light Chain zählen
    if pd.notnull(row.get("cdrl1_cluster")):
        canonical_entries.append({
            "PDB_ID": pdb_id,
            "Chain": chain,
            "CDR": "CDR-L1",
            "Canonical_Cluster": row["cdrl1_cluster"]
        })

    if pd.notnull(row.get("cdrl2_cluster")):
        canonical_entries.append({
            "PDB_ID": pdb_id,
            "Chain": chain,
            "CDR": "CDR-L2",
            "Canonical_Cluster": row["cdrl2_cluster"]
        })

    if pd.notnull(row.get("cdrl3_cluster")):
        canonical_entries.append({
            "PDB_ID": pdb_id,
            "Chain": chain,
            "CDR": "CDR-L3",
            "Canonical_Cluster": row["cdrl3_cluster"]
        })

# Umwandlung in DataFrame
canonical_df = pd.DataFrame(canonical_entries)


#funktion ausführen
vergleich_df = compare_esm_vs_canonical(canonical_df, esm_clusters_df)
print(vergleich_df)

#damit das ganze funktioniert muss auch esm_clusters_df in richtigem format sein